# RecoMart Data Preparation & Exploratory Data Analysis

## Overview
This notebook handles data cleaning, preprocessing, and exploratory data analysis for the RecoMart recommendation pipeline. It processes raw data with quality issues and prepares it for feature engineering.

## Objectives
1. **Data Cleaning**: Handle missing values, duplicates, and outliers
2. **Data Type Conversions**: Ensure proper data types for all columns
3. **Categorical Encoding**: Encode categorical variables
4. **Normalization**: Scale numerical features
5. **Exploratory Data Analysis**: Understand data distributions and patterns
6. **Data Quality Remediation**: Fix issues identified in validation

## Input Tables
* `recomart.raw.user_interactions` (with quality issues)
* `recomart.raw.transactions` (with quality issues)
* `recomart.raw.products` (with quality issues)

## Output Tables
* `recomart.clean.user_interactions_clean`
* `recomart.clean.transactions_clean`
* `recomart.clean.products_clean`

In [0]:
# Import required libraries
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import logging
import os

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Configure logging
log_dir = "/Workspace/Users/2025ae05415@wilp.bits-pilani.ac.in/RecoMart_Recommendation_Pipeline/logs"
os.makedirs(log_dir, exist_ok=True)
log_file = f"{log_dir}/preparation.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger('RecoMartPreparation')

# Unity Catalog configuration
CATALOG_NAME = 'recomart'
RAW_SCHEMA = 'raw'
CLEAN_SCHEMA = 'clean'

# Create clean schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{CLEAN_SCHEMA}")

logger.info("="*80)
logger.info("RecoMart Data Preparation Pipeline - Session Started")
logger.info(f"Timestamp: {datetime.now().isoformat()}")
logger.info("="*80)

print("✅ Setup complete - Data preparation framework initialized")
print(f"📁 Log file: {log_file}")

## 1. Load Raw Data & Assess Quality Issues

In [0]:
print("\n" + "="*80)
print("📥 LOADING RAW DATA")
print("="*80)

# Load raw tables
df_interactions_raw = spark.table(f"{CATALOG_NAME}.{RAW_SCHEMA}.user_interactions")
df_transactions_raw = spark.table(f"{CATALOG_NAME}.{RAW_SCHEMA}.transactions")
df_products_raw = spark.table(f"{CATALOG_NAME}.{RAW_SCHEMA}.products")

print(f"\n✅ Loaded raw data:")
print(f"  • User Interactions: {df_interactions_raw.count():,} records")
print(f"  • Transactions: {df_transactions_raw.count():,} records")
print(f"  • Products: {df_products_raw.count():,} records")

# Show quality issues summary
print("\n" + "="*80)
print("🔍 QUALITY ISSUES SUMMARY")
print("="*80)

def assess_quality(df, name):
    print(f"\n📊 {name}:")
    total = df.count()
    
    # Check for nulls in each column
    null_counts = []
    for col_name in df.columns:
        null_count = df.filter(F.col(col_name).isNull()).count()
        if null_count > 0:
            null_pct = (null_count / total) * 100
            null_counts.append((col_name, null_count, null_pct))
    
    if null_counts:
        print(f"  Missing Values:")
        for col, count, pct in sorted(null_counts, key=lambda x: x[1], reverse=True)[:5]:
            print(f"    • {col}: {count:,} ({pct:.2f}%)")
    else:
        print(f"  Missing Values: None")
    
    return null_counts

interactions_nulls = assess_quality(df_interactions_raw, "User Interactions")
transactions_nulls = assess_quality(df_transactions_raw, "Transactions")
products_nulls = assess_quality(df_products_raw, "Products")

## 2. Clean User Interactions

In [0]:
print("\n" + "="*80)
print("🧹 CLEANING USER INTERACTIONS")
print("="*80)

df_interactions = df_interactions_raw

# 1. Remove records with missing critical fields
initial_count = df_interactions.count()
df_interactions = df_interactions.filter(
    F.col("user_id").isNotNull() & 
    F.col("item_id").isNotNull() & 
    F.col("timestamp").isNotNull()
)
removed = initial_count - df_interactions.count()
print(f"\n  ✓ Removed {removed:,} records with missing critical fields")

# 2. Remove duplicates
initial_count = df_interactions.count()
df_interactions = df_interactions.dropDuplicates([
    'user_id', 'item_id', 'interaction_type', 'timestamp', 'session_id'
])
removed = initial_count - df_interactions.count()
print(f"  ✓ Removed {removed:,} duplicate records")

# 3. Fix invalid categorical values
# Valid device types
valid_devices = ['mobile', 'web', 'tablet']
df_interactions = df_interactions.withColumn(
    "device_type",
    F.when(F.col("device_type").isin(valid_devices), F.col("device_type"))
     .otherwise(F.lit("mobile"))  # Default to mobile for invalid values
)
print(f"  ✓ Fixed invalid device_type values (defaulted to 'mobile')")

# 4. Parse timestamp properly
df_interactions = df_interactions.withColumn(
    "timestamp_parsed",
    F.to_timestamp(F.col("timestamp"))
)

# 5. Add derived columns
df_interactions = df_interactions.withColumn(
    "interaction_hour", F.hour("timestamp_parsed")
).withColumn(
    "interaction_dayofweek", F.dayofweek("timestamp_parsed")
).withColumn(
    "cleaned_timestamp", F.current_timestamp()
)

final_count = df_interactions.count()
print(f"\n✅ User Interactions cleaned: {final_count:,} records")
print(f"   Retention rate: {(final_count/initial_count*100):.2f}%")

# Save to clean schema
df_interactions.write.format("delta").mode("overwrite")\
    .saveAsTable(f"{CATALOG_NAME}.{CLEAN_SCHEMA}.user_interactions_clean")

logger.info(f"Cleaned user interactions: {final_count} records")
print(f"\n💾 Saved to: {CATALOG_NAME}.{CLEAN_SCHEMA}.user_interactions_clean")

## 3. Clean Transactions

In [0]:
print("\n" + "="*80)
print("🧹 CLEANING TRANSACTIONS")
print("="*80)

df_transactions = df_transactions_raw

# 1. Remove records with missing critical fields
initial_count = df_transactions.count()
df_transactions = df_transactions.filter(
    F.col("transaction_id").isNotNull() & 
    F.col("user_id").isNotNull() & 
    F.col("item_id").isNotNull() &
    F.col("price").isNotNull()
)
removed = initial_count - df_transactions.count()
print(f"\n  ✓ Removed {removed:,} records with missing critical fields")

# 2. Remove duplicates based on transaction_id
initial_count = df_transactions.count()
df_transactions = df_transactions.dropDuplicates(["transaction_id"])
removed = initial_count - df_transactions.count()
print(f"  ✓ Removed {removed:,} duplicate transactions")

# 3. Fix invalid ratings (should be 1-5)
df_transactions = df_transactions.withColumn(
    "rating_cleaned",
    F.when((F.col("rating") >= 1) & (F.col("rating") <= 5), F.col("rating"))
     .when(F.col("rating") > 5, 5)  # Cap at 5
     .when(F.col("rating") < 1, 1)  # Floor at 1
     .otherwise(3)  # Default to 3 (neutral) for null/invalid
)
invalid_ratings = df_transactions.filter(
    (F.col("rating") < 1) | (F.col("rating") > 5) | F.col("rating").isNull()
).count()
print(f"  ✓ Fixed {invalid_ratings:,} invalid rating values")

# 4. Fix negative or zero prices
df_transactions = df_transactions.filter(F.col("price") > 0)
print(f"  ✓ Removed records with invalid prices (≤ 0)")

# 5. Fix invalid quantities
df_transactions = df_transactions.withColumn(
    "quantity_cleaned",
    F.when((F.col("quantity") >= 1) & (F.col("quantity") <= 100), F.col("quantity"))
     .otherwise(1)  # Default to 1 for invalid values
)

# 6. Fix invalid payment methods
valid_payment_methods = ['credit_card', 'debit_card', 'paypal', 'upi', 'wallet']
df_transactions = df_transactions.withColumn(
    "payment_method_cleaned",
    F.when(F.col("payment_method").isin(valid_payment_methods), F.col("payment_method"))
     .otherwise(F.lit("credit_card"))  # Default
)
print(f"  ✓ Fixed invalid payment methods")

# 7. Parse timestamp
df_transactions = df_transactions.withColumn(
    "timestamp_parsed",
    F.to_timestamp(F.col("timestamp"))
)

# 8. Calculate total amount
df_transactions = df_transactions.withColumn(
    "total_amount",
    F.col("price") * F.col("quantity_cleaned")
).withColumn(
    "cleaned_timestamp", F.current_timestamp()
)

final_count = df_transactions.count()
print(f"\n✅ Transactions cleaned: {final_count:,} records")

# Save to clean schema
df_transactions.write.format("delta").mode("overwrite")\
    .saveAsTable(f"{CATALOG_NAME}.{CLEAN_SCHEMA}.transactions_clean")

logger.info(f"Cleaned transactions: {final_count} records")
print(f"\n💾 Saved to: {CATALOG_NAME}.{CLEAN_SCHEMA}.transactions_clean")

## 4. Clean Products

In [0]:
print("\n" + "="*80)
print("🧹 CLEANING PRODUCTS")
print("="*80)

df_products = df_products_raw

# 1. Remove records with missing critical fields
initial_count = df_products.count()
df_products = df_products.filter(
    F.col("item_id").isNotNull() & 
    F.col("product_name").isNotNull() & 
    F.col("category").isNotNull() &
    F.col("price").isNotNull()
)
removed = initial_count - df_products.count()
print(f"\n  ✓ Removed {removed:,} records with missing critical fields")

# 2. Remove duplicates based on item_id (keep latest)
window_spec = Window.partitionBy("item_id").orderBy(F.col("ingestion_timestamp").desc())
df_products = df_products.withColumn("row_num", F.row_number().over(window_spec))\
    .filter(F.col("row_num") == 1)\
    .drop("row_num")
print(f"  ✓ Removed duplicate products (kept latest)")

# 3. Fix invalid prices
df_products = df_products.filter((F.col("price") > 0) & (F.col("price") < 100000))
print(f"  ✓ Removed records with invalid prices")

# 4. Fix out-of-range scores
df_products = df_products.withColumn(
    "popularity_score_cleaned",
    F.when((F.col("popularity_score") >= 0) & (F.col("popularity_score") <= 100), 
           F.col("popularity_score"))
     .when(F.col("popularity_score") > 100, 100.0)
     .when(F.col("popularity_score") < 0, 0.0)
     .otherwise(50.0)  # Default to median
)

df_products = df_products.withColumn(
    "sentiment_score_cleaned",
    F.when((F.col("sentiment_score") >= 0) & (F.col("sentiment_score") <= 1), 
           F.col("sentiment_score"))
     .when(F.col("sentiment_score") > 1, 1.0)
     .when(F.col("sentiment_score") < 0, 0.0)
     .otherwise(0.5)  # Default to neutral
)
print(f"  ✓ Fixed out-of-range popularity and sentiment scores")

# 5. Fix invalid stock status
valid_stock_status = ['in_stock', 'out_of_stock', 'limited_stock']
df_products = df_products.withColumn(
    "stock_status_cleaned",
    F.when(F.col("stock_status").isin(valid_stock_status), F.col("stock_status"))
     .otherwise(F.lit("in_stock"))  # Default
)
print(f"  ✓ Fixed invalid stock status values")

# 6. Handle missing sub_category (fill with 'Other')
df_products = df_products.withColumn(
    "sub_category_cleaned",
    F.when(F.col("sub_category").isNotNull(), F.col("sub_category"))
     .otherwise(F.lit("Other"))
)

# 7. Create price categories
df_products = df_products.withColumn(
    "price_category",
    F.when(F.col("price") < 50, "Budget")
     .when((F.col("price") >= 50) & (F.col("price") < 200), "Mid-Range")
     .when(F.col("price") >= 200, "Premium")
     .otherwise("Unknown")
).withColumn(
    "cleaned_timestamp", F.current_timestamp()
)

final_count = df_products.count()
print(f"\n✅ Products cleaned: {final_count:,} records")

# Save to clean schema
df_products.write.format("delta").mode("overwrite")\
    .saveAsTable(f"{CATALOG_NAME}.{CLEAN_SCHEMA}.products_clean")

logger.info(f"Cleaned products: {final_count} records")
print(f"\n💾 Saved to: {CATALOG_NAME}.{CLEAN_SCHEMA}.products_clean")

## 5. Exploratory Data Analysis

In [0]:
print("\n" + "="*80)
print("📊 EXPLORATORY DATA ANALYSIS")
print("="*80)

# Load cleaned data
df_int_clean = spark.table(f"{CATALOG_NAME}.{CLEAN_SCHEMA}.user_interactions_clean")
df_txn_clean = spark.table(f"{CATALOG_NAME}.{CLEAN_SCHEMA}.transactions_clean")
df_prod_clean = spark.table(f"{CATALOG_NAME}.{CLEAN_SCHEMA}.products_clean")

print("\n✅ Loaded cleaned datasets")

# ============================================================
# 1. USER INTERACTIONS ANALYSIS
# ============================================================
print("\n" + "-"*80)
print("📈 User Interactions Analysis")
print("-"*80)

# Interaction type distribution
int_type_dist = df_int_clean.groupBy("interaction_type")\
    .count()\
    .orderBy(F.desc("count"))

print("\nInteraction Type Distribution:")
int_type_dist.show()

# Convert to pandas for plotting
int_type_pd = int_type_dist.toPandas()

# Plot interaction types
plt.figure(figsize=(10, 6))
plt.bar(int_type_pd['interaction_type'], int_type_pd['count'], color='skyblue')
plt.title('Distribution of Interaction Types', fontsize=16, fontweight='bold')
plt.xlabel('Interaction Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
for i, v in enumerate(int_type_pd['count']):
    plt.text(i, v + 50, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# Device type distribution
device_dist = df_int_clean.groupBy("device_type")\
    .count()\
    .orderBy(F.desc("count"))

print("\nDevice Type Distribution:")
device_dist.show()

# Plot device types
device_pd = device_dist.toPandas()
plt.figure(figsize=(10, 6))
plt.pie(device_pd['count'], labels=device_pd['device_type'], autopct='%1.1f%%', startangle=90)
plt.title('Device Type Distribution', fontsize=16, fontweight='bold')
plt.axis('equal')
plt.tight_layout()
plt.show()

# Hourly interaction pattern
hourly_pattern = df_int_clean.groupBy("interaction_hour")\
    .count()\
    .orderBy("interaction_hour")

print("\nHourly Interaction Pattern:")
hourly_pattern.show(24)

hourly_pd = hourly_pattern.toPandas()
plt.figure(figsize=(14, 6))
plt.plot(hourly_pd['interaction_hour'], hourly_pd['count'], marker='o', linewidth=2, markersize=8)
plt.title('Interaction Pattern by Hour of Day', fontsize=16, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Interactions')
plt.grid(True, alpha=0.3)
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# 2. TRANSACTIONS ANALYSIS
# ============================================================
print("\n" + "-"*80)
print("💰 Transactions Analysis")
print("-"*80)

# Transaction statistics
txn_stats = df_txn_clean.select(
    F.count("*").alias("total_transactions"),
    F.sum("total_amount").alias("total_revenue"),
    F.avg("total_amount").alias("avg_transaction_value"),
    F.min("total_amount").alias("min_transaction"),
    F.max("total_amount").alias("max_transaction"),
    F.countDistinct("user_id").alias("unique_users"),
    F.countDistinct("item_id").alias("unique_products")
)

print("\nTransaction Statistics:")
txn_stats.show(truncate=False)

# Rating distribution
rating_dist = df_txn_clean.groupBy("rating_cleaned")\
    .count()\
    .orderBy("rating_cleaned")

print("\nRating Distribution:")
rating_dist.show()

rating_pd = rating_dist.toPandas()
plt.figure(figsize=(10, 6))
plt.bar(rating_pd['rating_cleaned'].astype(str), rating_pd['count'], color='coral')
plt.title('Distribution of Product Ratings', fontsize=16, fontweight='bold')
plt.xlabel('Rating')
plt.ylabel('Count')
for i, v in enumerate(rating_pd['count']):
    plt.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# Payment method distribution
payment_dist = df_txn_clean.groupBy("payment_method_cleaned")\
    .count()\
    .orderBy(F.desc("count"))

print("\nPayment Method Distribution:")
payment_dist.show()

# Top spending users
top_users = df_txn_clean.groupBy("user_id")\
    .agg(
        F.sum("total_amount").alias("total_spent"),
        F.count("*").alias("num_transactions")
    )\
    .orderBy(F.desc("total_spent"))\
    .limit(10)

print("\nTop 10 Spending Users:")
top_users.show()

In [0]:
# ============================================================
# 3. PRODUCTS ANALYSIS
# ============================================================
print("\n" + "-"*80)
print("🛍️ Products Analysis")
print("-"*80)

# Product statistics
prod_stats = df_prod_clean.select(
    F.count("*").alias("total_products"),
    F.avg("price").alias("avg_price"),
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.countDistinct("category").alias("unique_categories"),
    F.countDistinct("brand").alias("unique_brands")
)

print("\nProduct Statistics:")
prod_stats.show(truncate=False)

# Category distribution
category_dist = df_prod_clean.groupBy("category")\
    .count()\
    .orderBy(F.desc("count"))

print("\nCategory Distribution:")
category_dist.show()

cat_pd = category_dist.toPandas()
plt.figure(figsize=(12, 6))
plt.barh(cat_pd['category'], cat_pd['count'], color='lightgreen')
plt.title('Product Distribution by Category', fontsize=16, fontweight='bold')
plt.xlabel('Count')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

# Price distribution by category
price_by_cat = df_prod_clean.groupBy("category")\
    .agg(
        F.avg("price").alias("avg_price"),
        F.min("price").alias("min_price"),
        F.max("price").alias("max_price")
    )\
    .orderBy(F.desc("avg_price"))

print("\nPrice Statistics by Category:")
price_by_cat.show(truncate=False)

# Price category distribution
price_cat_dist = df_prod_clean.groupBy("price_category")\
    .count()\
    .orderBy("price_category")

print("\nPrice Category Distribution:")
price_cat_dist.show()

# Stock status distribution
stock_dist = df_prod_clean.groupBy("stock_status_cleaned")\
    .count()\
    .orderBy(F.desc("count"))

print("\nStock Status Distribution:")
stock_dist.show()

stock_pd = stock_dist.toPandas()
plt.figure(figsize=(10, 6))
colors = ['#90EE90', '#FFB6C1', '#FFD700']
plt.bar(stock_pd['stock_status_cleaned'], stock_pd['count'], color=colors)
plt.title('Product Stock Status Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Stock Status')
plt.ylabel('Count')
plt.xticks(rotation=45)
for i, v in enumerate(stock_pd['count']):
    plt.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# 4. USER-ITEM INTERACTION MATRIX SPARSITY
# ============================================================
print("\n" + "-"*80)
print("🔍 Interaction Matrix Sparsity Analysis")
print("-"*80)

num_users = df_int_clean.select("user_id").distinct().count()
num_items = df_int_clean.select("item_id").distinct().count()
num_interactions = df_int_clean.count()

matrix_size = num_users * num_items
sparsity = 1 - (num_interactions / matrix_size)

print(f"\nUser-Item Matrix Dimensions:")
print(f"  • Number of users: {num_users:,}")
print(f"  • Number of items: {num_items:,}")
print(f"  • Matrix size: {matrix_size:,} cells")
print(f"  • Number of interactions: {num_interactions:,}")
print(f"  • Sparsity: {sparsity*100:.2f}%")
print(f"  • Density: {(1-sparsity)*100:.2f}%")

# User activity distribution
user_activity = df_int_clean.groupBy("user_id")\
    .count()\
    .withColumnRenamed("count", "num_interactions")

activity_stats = user_activity.select(
    F.avg("num_interactions").alias("avg_interactions_per_user"),
    F.min("num_interactions").alias("min_interactions"),
    F.max("num_interactions").alias("max_interactions"),
    F.percentile_approx("num_interactions", 0.5).alias("median_interactions")
)

print("\nUser Activity Statistics:")
activity_stats.show(truncate=False)

# Item popularity distribution
item_popularity = df_int_clean.groupBy("item_id")\
    .count()\
    .withColumnRenamed("count", "num_interactions")

popularity_stats = item_popularity.select(
    F.avg("num_interactions").alias("avg_interactions_per_item"),
    F.min("num_interactions").alias("min_interactions"),
    F.max("num_interactions").alias("max_interactions"),
    F.percentile_approx("num_interactions", 0.5).alias("median_interactions")
)

print("\nItem Popularity Statistics:")
popularity_stats.show(truncate=False)

print("\n" + "="*80)
logger.info("="*80)
logger.info("Data Preparation Pipeline - Session Completed")
logger.info("="*80)

print("\n✅ Data preparation and EDA completed!")
print(f"📝 Full logs available at: {log_file}")